# Practical 11 - Index Templates, Aliases, Lifecycle and Optimization

**Dataset:** 100,000 deterministic synthetic events in `p11-baseline-events`

**Stack:** Elasticsearch 9.2.0, Python `elasticsearch` 9.5.1, kernel `es`

This practical covers the operational side of an Elasticsearch index:

1. Component templates and an index template.
2. Creating an index from the template.
3. Aliases.
4. Atomic alias switching.
5. Reindexing into a new index.
6. Shard configuration.
7. Refresh interval tuning.
8. Indexing performance comparison.
9. Slow-query investigation.
10. Index lifecycle management (ILM).
11. Rollover and data streams.
12. Index and node statistics.

The assignment compares a deliberately less-optimal baseline index with an
optimized index containing the same 100,000 documents.

## Setup

The notebook recreates every Practical 11 index at the start, so it is safe to
run repeatedly. The baseline intentionally uses dynamic mapping and five shards.
The optimized index uses one shard, an explicit mapping, `best_compression`, and
a force-merge to one segment.

In [1]:
import json
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

from elasticsearch import Elasticsearch, NotFoundError, helpers

from generate_data import FIELDS, build_records

HERE = Path.cwd()
assert HERE.name == "Practical_11", f"Run this notebook from Practical_11, not {HERE}"

ARTIFACTS = HERE / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

ES_URL = "http://127.0.0.1:9200"
TOTAL = 100_000
SMALL_TOTAL = 20_000
QUERY_ITERATIONS = 20

BASELINE = "p11-baseline-events"
OPTIMIZED = "p11-events-optimized"
REINDEXED = "p11-events-reindexed"
READ_ALIAS = "p11-events-read"

SHARDS_1 = "p11-shards-1"
SHARDS_3 = "p11-shards-3"
REFRESH_1S = "p11-refresh-1s"
REFRESH_OFF = "p11-refresh-off"

EVENTS_TEMPLATE = "p11-events-template"
SETTINGS_COMPONENT = "p11-events-settings"
MAPPINGS_COMPONENT = "p11-events-mappings"

ILM_POLICY = "p11-events-ilm"
ILM_DEMO = "p11-events-ilm-demo"
ILM_ALIAS = "p11-events-ilm-write"

ROLLOVER_1 = "p11-rollover-000001"
ROLLOVER_2 = "p11-rollover-000002"
ROLLOVER_ALIAS = "p11-rollover-write"

LOGS_TEMPLATE = "p11-logs-template"
DATA_STREAM = "p11-logs"

MAPPING = json.loads((HERE / "mapping.json").read_text(encoding="utf-8"))
SETTINGS = MAPPING["settings"]
MAPPINGS = MAPPING["mappings"]

es = Elasticsearch(ES_URL, request_timeout=120)
info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])
print("dataset:", TOTAL, "baseline documents")


def plain(value):
    if hasattr(value, "items"):
        return {key: plain(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(item) for item in value]
    return value


def delete_index(name):
    if es.indices.exists(index=name):
        es.indices.delete(index=name)


def index_count(index):
    return int(es.count(index=index)["count"])


def index_setting(index, key):
    settings = es.indices.get_settings(index=index)[index]["settings"]["index"]
    return settings.get(key)


def bulk_load(index, count=TOTAL, chunk_size=5000):
    started = time.perf_counter()
    loaded, errors = helpers.bulk(
        es,
        (
            {"_index": index, "_id": record["event_id"], "_source": record}
            for record in build_records(count=count)
        ),
        chunk_size=chunk_size,
        raise_on_error=False,
    )
    seconds = time.perf_counter() - started
    return {
        "documents": int(loaded),
        "errors": len(errors),
        "seconds": round(seconds, 3),
        "docs_per_second": round(loaded / seconds, 1),
    }


def index_metrics(index):
    es.indices.flush(index=index)
    stats = es.indices.stats(
        index=index,
        metric=["docs", "store", "segments", "search", "indexing"],
        level="indices",
    )
    primaries = stats["indices"][index]["primaries"]
    return {
        "documents": int(primaries["docs"]["count"]),
        "primary_store_bytes": int(primaries["store"]["size_in_bytes"]),
        "segments": int(primaries["segments"]["count"]),
        "search_query_total": int(primaries["search"]["query_total"]),
        "search_query_time_ms": int(primaries["search"]["query_time_in_millis"]),
        "indexing_index_total": int(primaries["indexing"]["index_total"]),
        "indexing_index_time_ms": int(primaries["indexing"]["index_time_in_millis"]),
    }

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node
dataset: 100000 baseline documents


In [2]:
try:
    es.indices.delete_data_stream(name=DATA_STREAM)
except NotFoundError:
    pass

for name in [
    BASELINE,
    OPTIMIZED,
    REINDEXED,
    SHARDS_1,
    SHARDS_3,
    REFRESH_1S,
    REFRESH_OFF,
    ILM_DEMO,
    ROLLOVER_1,
    ROLLOVER_2,
]:
    delete_index(name)

print("Old Practical 11 indices removed.")
print("Elasticsearch cluster health:", es.cluster.health()["status"])

Old Practical 11 indices removed.
Elasticsearch cluster health: yellow


## Task 1 - Create component templates and an index template

Component templates are reusable building blocks. The settings component holds
shard, replica, refresh, and codec settings. The mappings component holds the
explicit field mapping.

The index template composes both components and applies to `p11-events-*`.
Component templates use the `cluster` API namespace; index templates use the
`indices` namespace.

In [3]:
es.cluster.put_component_template(
    name=SETTINGS_COMPONENT,
    template={"settings": SETTINGS},
)
es.cluster.put_component_template(
    name=MAPPINGS_COMPONENT,
    template={"mappings": MAPPINGS},
)
es.indices.put_index_template(
    name=EVENTS_TEMPLATE,
    index_patterns=["p11-events-*"],
    composed_of=[SETTINGS_COMPONENT, MAPPINGS_COMPONENT],
    priority=200,
)

component_snapshot = {"component_templates": []}
for component_name in [SETTINGS_COMPONENT, MAPPINGS_COMPONENT]:
    component_snapshot["component_templates"].extend(
        plain(es.cluster.get_component_template(name=component_name))["component_templates"]
    )
template_snapshot = plain(es.indices.get_index_template(name=EVENTS_TEMPLATE))

component_names = [item["name"] for item in component_snapshot["component_templates"]]
template_names = [item["name"] for item in template_snapshot["index_templates"]]

print("component templates:", component_names)
print("index template:", template_names)
print("index pattern:", template_snapshot["index_templates"][0]["index_template"]["index_patterns"])
print("composed of:", template_snapshot["index_templates"][0]["index_template"]["composed_of"])

assert set(component_names) == {SETTINGS_COMPONENT, MAPPINGS_COMPONENT}
assert template_names == [EVENTS_TEMPLATE]
assert template_snapshot["index_templates"][0]["index_template"]["index_patterns"] == ["p11-events-*"]

component templates: ['p11-events-settings', 'p11-events-mappings']
index template: ['p11-events-template']
index pattern: ['p11-events-*']
composed of: ['p11-events-settings', 'p11-events-mappings']


## Task 2 - Create an index from the template

Creating `p11-events-optimized` without a body lets the index template supply
the settings and mappings. The resolved index should have one shard, zero
replicas, `best_compression`, and an explicit `strict` mapping.

In [4]:
delete_index(OPTIMIZED)
es.indices.create(index=OPTIMIZED)

optimized_settings = es.indices.get_settings(index=OPTIMIZED)[OPTIMIZED]["settings"]["index"]
optimized_mapping = es.indices.get_mapping(index=OPTIMIZED)[OPTIMIZED]["mappings"]

print("shards:", optimized_settings["number_of_shards"])
print("replicas:", optimized_settings["number_of_replicas"])
print("codec:", optimized_settings.get("codec"))
print("refresh interval:", optimized_settings.get("refresh_interval"))
print("dynamic mapping:", optimized_mapping.get("dynamic"))
print("field types:", {name: prop["type"] for name, prop in optimized_mapping["properties"].items()})

assert int(optimized_settings["number_of_shards"]) == 1
assert int(optimized_settings["number_of_replicas"]) == 0
assert optimized_settings.get("codec") == "best_compression"
assert optimized_mapping.get("dynamic") == "strict"
assert optimized_mapping["properties"]["description"]["norms"] is False

shards: 1
replicas: 0
codec: best_compression
refresh interval: 30s
dynamic mapping: strict
field types: {'amount': 'integer', 'category': 'keyword', 'created_at': 'date', 'customer_id': 'keyword', 'description': 'text', 'event_id': 'keyword', 'region': 'keyword', 'sequence': 'integer', 'status': 'keyword'}


## Baseline dataset - five shards and dynamic mapping

The baseline is deliberately not optimal. It uses five primary shards on a
single node and lets Elasticsearch infer every field from the first documents.
The inferred string fields receive a `.keyword` subfield, which makes the
baseline mapping larger than the optimized explicit mapping.

In [5]:
delete_index(BASELINE)
es.indices.create(
    index=BASELINE,
    settings={
        "number_of_shards": 5,
        "number_of_replicas": 0,
        "refresh_interval": "1s",
    },
)

baseline_load = bulk_load(BASELINE)
es.indices.refresh(index=BASELINE)
baseline_count = index_count(BASELINE)
baseline_metrics = index_metrics(BASELINE)
baseline_mapping = es.indices.get_mapping(index=BASELINE)[BASELINE]["mappings"]

print("baseline load:", baseline_load)
print("baseline documents:", f"{baseline_count:,}")
print("baseline shards:", index_setting(BASELINE, "number_of_shards"))
print("status mapping:", baseline_mapping["properties"]["status"])
print("description mapping:", baseline_mapping["properties"]["description"])
print("baseline metrics:", baseline_metrics)

assert baseline_load["documents"] == TOTAL
assert baseline_load["errors"] == 0
assert baseline_count == TOTAL
assert int(index_setting(BASELINE, "number_of_shards")) == 5
assert "keyword" in baseline_mapping["properties"]["status"]["fields"]
assert "keyword" in baseline_mapping["properties"]["description"]["fields"]

baseline load: {'documents': 100000, 'errors': 0, 'seconds': 43.798, 'docs_per_second': 2283.2}
baseline documents: 100,000
baseline shards: 5
status mapping: {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}
description mapping: {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}
baseline metrics: {'documents': 100000, 'primary_store_bytes': 19375495, 'segments': 23, 'search_query_total': 5, 'search_query_time_ms': 0, 'indexing_index_total': 100000, 'indexing_index_time_ms': 30480}


## Task 3 - Create an alias

An alias is a stable name in front of one or more indices. Applications should
query the alias rather than a concrete index name. The `_read` alias starts on
the baseline index.

In [6]:
delete_index(OPTIMIZED)
es.indices.create(index=OPTIMIZED)

es.indices.put_alias(index=BASELINE, name=READ_ALIAS)
alias_before = plain(es.indices.get_alias(name=READ_ALIAS))
baseline_alias_count = int(es.count(index=READ_ALIAS)["count"])

print("alias mapping:", {index: list(names) for index, names in alias_before.items()})
print("documents through alias:", f"{baseline_alias_count:,}")

assert BASELINE in alias_before
assert READ_ALIAS in alias_before[BASELINE]["aliases"]
assert baseline_alias_count == TOTAL

alias mapping: {'p11-baseline-events': ['aliases']}
documents through alias: 100,000


## Task 4 - Load the optimized index and switch the alias atomically

The optimized index is loaded with refresh disabled. That removes repeated
Lucene refreshes from the critical indexing path. After loading, refresh is
restored and the index is force-merged to one segment.

The alias switch is a single `_aliases` request containing both the remove and
add actions. Elasticsearch applies it atomically, so readers never see a
moment where the alias is missing.

In [7]:
es.indices.put_settings(index=OPTIMIZED, settings={"index.refresh_interval": "-1"})
optimized_load = bulk_load(OPTIMIZED)
es.indices.put_settings(index=OPTIMIZED, settings={"index.refresh_interval": "30s"})
es.indices.refresh(index=OPTIMIZED)
optimized_count = index_count(OPTIMIZED)
optimized_before_merge = index_metrics(OPTIMIZED)

force_merge_started = time.perf_counter()
es.indices.forcemerge(
    index=OPTIMIZED,
    max_num_segments=1,
    flush=True,
    wait_for_completion=True,
)
force_merge_seconds = time.perf_counter() - force_merge_started
es.indices.refresh(index=OPTIMIZED)
optimized_metrics = index_metrics(OPTIMIZED)

print("optimized load:", optimized_load)
print("optimized documents:", f"{optimized_count:,}")
print("before merge segments:", optimized_before_merge["segments"])
print("force merge seconds:", round(force_merge_seconds, 3))
print("after merge segments:", optimized_metrics["segments"])
print("optimized metrics:", optimized_metrics)

assert optimized_load["documents"] == TOTAL
assert optimized_load["errors"] == 0
assert optimized_count == TOTAL
assert optimized_metrics["segments"] == 1

optimized load: {'documents': 100000, 'errors': 0, 'seconds': 14.66, 'docs_per_second': 6821.3}
optimized documents: 100,000
before merge segments: 1
force merge seconds: 0.005
after merge segments: 1
optimized metrics: {'documents': 100000, 'primary_store_bytes': 3962288, 'segments': 1, 'search_query_total': 1, 'search_query_time_ms': 0, 'indexing_index_total': 100000, 'indexing_index_time_ms': 5373}


In [8]:
es.indices.update_aliases(
    actions=[
        {"remove": {"index": BASELINE, "alias": READ_ALIAS}},
        {"add": {"index": OPTIMIZED, "alias": READ_ALIAS}},
    ]
)
alias_after = plain(es.indices.get_alias(name=READ_ALIAS))
optimized_alias_count = int(es.count(index=READ_ALIAS)["count"])

print("alias mapping:", {index: list(names) for index, names in alias_after.items()})
print("documents through alias:", f"{optimized_alias_count:,}")

assert OPTIMIZED in alias_after
assert BASELINE not in alias_after
assert READ_ALIAS in alias_after[OPTIMIZED]["aliases"]
assert optimized_alias_count == TOTAL

alias mapping: {'p11-events-optimized': ['aliases']}
documents through alias: 100,000


## Task 5 - Reindex into a new index

Reindexing copies documents from the baseline into another index. The
destination index is created from the `p11-events-*` template, so it receives
the same optimized mapping and one-shard settings.

In [9]:
delete_index(REINDEXED)
es.indices.create(index=REINDEXED)

reindex_started = time.perf_counter()
reindex_response = es.reindex(
    source={"index": BASELINE},
    dest={"index": REINDEXED},
    wait_for_completion=True,
    refresh=True,
)
reindex_seconds = time.perf_counter() - reindex_started
reindexed_count = index_count(REINDEXED)
reindex_response = plain(reindex_response)

print("reindex result:", reindex_response)
print("reindex seconds:", round(reindex_seconds, 3))
print("reindexed documents:", f"{reindexed_count:,}")
print("reindexed metrics:", index_metrics(REINDEXED))

assert reindexed_count == TOTAL
assert reindex_response["created"] == TOTAL
assert reindex_response["failures"] == []
assert reindex_seconds > 0

reindex result: {'took': 13045, 'timed_out': False, 'total': 100000, 'updated': 0, 'created': 100000, 'deleted': 0, 'batches': 100, 'version_conflicts': 0, 'noops': 0, 'retries': {'bulk': 0, 'search': 0}, 'throttled_millis': 0, 'requests_per_second': -1.0, 'throttled_until_millis': 0, 'failures': []}
reindex seconds: 13.047
reindexed documents: 100,000


reindexed metrics: {'documents': 100000, 'primary_store_bytes': 10019210, 'segments': 1, 'search_query_total': 1, 'search_query_time_ms': 0, 'indexing_index_total': 100000, 'indexing_index_time_ms': 3189}


## Task 6 - Compare one shard with three shards

Both indices contain the same 20,000 documents and use the same explicit
mapping. Only the shard count differs. On a single node, extra shards add
Lucene segment overhead and a coordination phase without adding hardware
parallelism.

In [10]:
def create_mapped_index(index, shards, count=SMALL_TOTAL):
    delete_index(index)
    es.indices.create(
        index=index,
        settings={
            "number_of_shards": shards,
            "number_of_replicas": 0,
            "refresh_interval": "1s",
        },
        mappings=MAPPINGS,
    )
    load = bulk_load(index, count=count)
    es.indices.refresh(index=index)
    count_after = index_count(index)
    metrics = index_metrics(index)
    assert load["documents"] == count
    assert load["errors"] == 0
    assert count_after == count
    return load, metrics


shard_1_load, shard_1_metrics = create_mapped_index(SHARDS_1, 1)
shard_3_load, shard_3_metrics = create_mapped_index(SHARDS_3, 3)

print("one shard:", shard_1_load, shard_1_metrics)
print("three shards:", shard_3_load, shard_3_metrics)
print("configured shards:", index_setting(SHARDS_1, "number_of_shards"), index_setting(SHARDS_3, "number_of_shards"))

assert int(index_setting(SHARDS_1, "number_of_shards")) == 1
assert int(index_setting(SHARDS_3, "number_of_shards")) == 3

one shard: {'documents': 20000, 'errors': 0, 'seconds': 1.671, 'docs_per_second': 11970.8} {'documents': 20000, 'primary_store_bytes': 2610147, 'segments': 2, 'search_query_total': 1, 'search_query_time_ms': 0, 'indexing_index_total': 20000, 'indexing_index_time_ms': 401}
three shards: {'documents': 20000, 'errors': 0, 'seconds': 1.503, 'docs_per_second': 13304.5} {'documents': 20000, 'primary_store_bytes': 919386, 'segments': 6, 'search_query_total': 3, 'search_query_time_ms': 0, 'indexing_index_total': 20000, 'indexing_index_time_ms': 440}
configured shards: 1 3


The same combined filter is measured on both shard layouts. Warm-up requests
are excluded, and the reported values are wall-clock p50 and p95 over 20 runs.

In [11]:
def percentile(values, percentage):
    ordered = sorted(values)
    index = min(len(ordered) - 1, int(round((percentage / 100) * (len(ordered) - 1))))
    return ordered[index]


def measure_query(index, query, iterations=QUERY_ITERATIONS):
    for _ in range(3):
        es.search(index=index, query=query, size=10, track_total_hits=False)

    wall_ms = []
    took_ms = []
    for _ in range(iterations):
        started = time.perf_counter()
        response = es.search(index=index, query=query, size=10, track_total_hits=False)
        wall_ms.append((time.perf_counter() - started) * 1000)
        took_ms.append(response["took"])

    return {
        "iterations": iterations,
        "p50_ms": round(percentile(wall_ms, 50), 3),
        "p95_ms": round(percentile(wall_ms, 95), 3),
        "mean_took_ms": round(statistics.mean(took_ms), 3),
    }


SHARD_QUERY = {
    "bool": {
        "filter": [
            {"term": {"status": "completed"}},
            {"range": {"amount": {"gte": 100_000}}},
        ]
    }
}

shard_1_query = measure_query(SHARDS_1, SHARD_QUERY)
shard_3_query = measure_query(SHARDS_3, SHARD_QUERY)
shard_1_count = int(es.count(index=SHARDS_1, query=SHARD_QUERY)["count"])
shard_3_count = int(es.count(index=SHARDS_3, query=SHARD_QUERY)["count"])

print("one shard query:", shard_1_query, "count:", shard_1_count)
print("three shards query:", shard_3_query, "count:", shard_3_count)
print("store bytes:", shard_1_metrics["primary_store_bytes"], shard_3_metrics["primary_store_bytes"])

assert shard_1_count == shard_3_count
assert shard_1_count > 0

one shard query: {'iterations': 20, 'p50_ms': 51.561, 'p95_ms': 58.847, 'mean_took_ms': 11.05} count: 3025
three shards query: {'iterations': 20, 'p50_ms': 52.731, 'p95_ms': 61.219, 'mean_took_ms': 4.3} count: 3025
store bytes: 2610147 919386


## Task 7 - Compare refresh intervals

The refresh interval controls how often new documents become searchable. A
short interval costs Lucene work during indexing; `-1` disables automatic
refresh until the load is complete. The same 20,000 documents are loaded into
both indices.

In [12]:
def create_refresh_index(index, interval, count=SMALL_TOTAL):
    delete_index(index)
    es.indices.create(
        index=index,
        settings={
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "refresh_interval": interval,
        },
        mappings=MAPPINGS,
    )
    load = bulk_load(index, count=count)
    if interval == "-1":
        es.indices.put_settings(index=index, settings={"index.refresh_interval": "1s"})
    es.indices.refresh(index=index)
    count_after = index_count(index)
    assert load["documents"] == count
    assert load["errors"] == 0
    assert count_after == count
    return load


refresh_1s_load = create_refresh_index(REFRESH_1S, "1s")
refresh_off_load = create_refresh_index(REFRESH_OFF, "-1")

print("refresh 1s:", refresh_1s_load)
print("refresh off:", refresh_off_load)
print("refresh speedup:", round(refresh_off_load["docs_per_second"] / refresh_1s_load["docs_per_second"], 3), "x")
print("configured refresh after load:", index_setting(REFRESH_1S, "refresh_interval"), index_setting(REFRESH_OFF, "refresh_interval"))

assert refresh_1s_load["documents"] == SMALL_TOTAL
assert refresh_off_load["documents"] == SMALL_TOTAL

refresh 1s: {'documents': 20000, 'errors': 0, 'seconds': 3.418, 'docs_per_second': 5851.9}
refresh off: {'documents': 20000, 'errors': 0, 'seconds': 3.142, 'docs_per_second': 6365.3}
refresh speedup: 1.088 x
configured refresh after load: 1s 1s


## Task 8 - Compare indexing performance

This table combines the two 20,000-document refresh tests with the two
100,000-document assignment indices. The baseline uses dynamic mapping, five
shards, and default refresh behavior. The optimized index uses explicit
mapping, one shard, and refresh disabled during the bulk load.

In [13]:
INDEXING_ROWS = [
    {"index": BASELINE, "documents": TOTAL, **baseline_load},
    {"index": OPTIMIZED, "documents": TOTAL, **optimized_load},
    {"index": REFRESH_1S, "documents": SMALL_TOTAL, **refresh_1s_load},
    {"index": REFRESH_OFF, "documents": SMALL_TOTAL, **refresh_off_load},
]

for row in INDEXING_ROWS:
    print(
        f"{row['index']:22s} {row['documents']:>7,} docs  "
        f"{row['seconds']:>7.3f}s  {row['docs_per_second']:>10,.1f} docs/s"
    )

assert all(row["errors"] == 0 for row in INDEXING_ROWS)

p11-baseline-events    100,000 docs   43.798s     2,283.2 docs/s
p11-events-optimized   100,000 docs   14.660s     6,821.3 docs/s
p11-refresh-1s          20,000 docs    3.418s     5,851.9 docs/s
p11-refresh-off         20,000 docs    3.142s     6,365.3 docs/s


## Assignment - Baseline versus optimized index

Both assignment indices contain all 100,000 documents. The query benchmark uses
the same logical filters, with `.keyword` field names for the dynamically
mapped baseline and direct `keyword` fields for the optimized index.

The first measured request set warms the caches. The reported values are p50
and p95 wall-clock latency over 20 measured requests per query.

In [14]:
def make_queries(fields):
    return {
        "status_term": {"term": {fields["status"]: "completed"}},
        "region_term": {"term": {fields["region"]: "North"}},
        "amount_range": {"range": {"amount": {"gte": 100_000, "lte": 160_000}}},
        "date_range": {
            "range": {
                "created_at": {
                    "gte": "2024-01-01",
                    "lt": "2024-02-01",
                }
            }
        },
        "description_match": {"match": {"description": "payment"}},
        "combined_filters": {
            "bool": {
                "filter": [
                    {"term": {fields["status"]: "completed"}},
                    {"range": {"amount": {"gte": 100_000}}},
                ]
            }
        },
    }


BASELINE_QUERIES = make_queries({"status": "status.keyword", "region": "region.keyword"})
OPTIMIZED_QUERIES = make_queries({"status": "status", "region": "region"})

baseline_counts = {
    name: int(es.count(index=BASELINE, query=query)["count"])
    for name, query in BASELINE_QUERIES.items()
}
optimized_counts = {
    name: int(es.count(index=OPTIMIZED, query=query)["count"])
    for name, query in OPTIMIZED_QUERIES.items()
}

query_comparison = {}
for name in BASELINE_QUERIES:
    baseline_measurement = measure_query(BASELINE, BASELINE_QUERIES[name])
    optimized_measurement = measure_query(OPTIMIZED, OPTIMIZED_QUERIES[name])
    p50_change = (
        round(
            (baseline_measurement["p50_ms"] - optimized_measurement["p50_ms"])
            / baseline_measurement["p50_ms"]
            * 100,
            2,
        )
        if baseline_measurement["p50_ms"]
        else None
    )
    query_comparison[name] = {
        "baseline_count": baseline_counts[name],
        "optimized_count": optimized_counts[name],
        "baseline": baseline_measurement,
        "optimized": optimized_measurement,
        "p50_latency_reduction_percent": p50_change,
    }
    print(
        f"{name:20s} count={baseline_counts[name]:>6,}  "
        f"baseline p50={baseline_measurement['p50_ms']:>7.3f}ms  "
        f"optimized p50={optimized_measurement['p50_ms']:>7.3f}ms  "
        f"reduction={p50_change}%"
    )

for name in BASELINE_QUERIES:
    assert baseline_counts[name] == optimized_counts[name]

status_term          count=25,184  baseline p50= 51.035ms  optimized p50= 62.925ms  reduction=-23.3%


region_term          count=19,902  baseline p50= 49.099ms  optimized p50= 49.212ms  reduction=-0.23%


amount_range         count=23,910  baseline p50= 47.848ms  optimized p50= 52.452ms  reduction=-9.62%


date_range           count= 3,100  baseline p50= 53.832ms  optimized p50= 61.152ms  reduction=-13.6%


description_match    count=20,151  baseline p50= 53.512ms  optimized p50= 51.920ms  reduction=2.98%


combined_filters     count=15,020  baseline p50= 54.131ms  optimized p50= 70.115ms  reduction=-29.53%


In [15]:
baseline_docs_per_second = baseline_load["docs_per_second"]
optimized_docs_per_second = optimized_load["docs_per_second"]
baseline_store_bytes = baseline_metrics["primary_store_bytes"]
optimized_store_bytes = optimized_metrics["primary_store_bytes"]
store_reduction = round(
    (baseline_store_bytes - optimized_store_bytes) / baseline_store_bytes * 100, 2
)
indexing_change = round(
    (optimized_docs_per_second - baseline_docs_per_second)
    / baseline_docs_per_second
    * 100,
    2,
)

print("baseline index:", baseline_metrics)
print("optimized index:", optimized_metrics)
print("indexing throughput change:", indexing_change, "%")
print("primary store reduction:", store_reduction, "%")
print("segments:", baseline_metrics["segments"], "->", optimized_metrics["segments"])

assert optimized_metrics["documents"] == TOTAL
assert baseline_metrics["documents"] == TOTAL

baseline index: {'documents': 100000, 'primary_store_bytes': 19375495, 'segments': 23, 'search_query_total': 5, 'search_query_time_ms': 0, 'indexing_index_total': 100000, 'indexing_index_time_ms': 30480}
optimized index: {'documents': 100000, 'primary_store_bytes': 3962288, 'segments': 1, 'search_query_total': 1, 'search_query_time_ms': 0, 'indexing_index_total': 100000, 'indexing_index_time_ms': 5373}
indexing throughput change: 198.76 %
primary store reduction: 79.55 %
segments: 23 -> 1


## Task 9 - Investigate a slow query

A script query evaluates a script for every candidate document. The equivalent
`range` filter can use the numeric index structure and the filter cache. Both
return the same number of documents. Profiling exposes the query type and
per-phase time inside Elasticsearch.

In [16]:
script_query = {
    "script": {
        "script": {
            "source": "doc['amount'].value >= params.min",
            "params": {"min": 100_000},
        }
    }
}
range_query = {"range": {"amount": {"gte": 100_000}}}

script_count = int(es.count(index=OPTIMIZED, query=script_query)["count"])
range_count = int(es.count(index=OPTIMIZED, query=range_query)["count"])

script_profile_response = es.search(
    index=OPTIMIZED,
    query=script_query,
    size=0,
    profile=True,
    track_total_hits=True,
)
range_profile_response = es.search(
    index=OPTIMIZED,
    query=range_query,
    size=0,
    profile=True,
    track_total_hits=True,
)


def profile_summary(response):
    entries = []
    for shard in response.get("profile", {}).get("shards", []):
        for search in shard.get("searches", []):
            for query in search.get("query", []):
                entries.append(
                    {
                        "type": query.get("type"),
                        "description": query.get("description"),
                        "time_in_nanos": query.get("time_in_nanos"),
                    }
                )
    return {
        "took_ms": response["took"],
        "shards": response["_shards"],
        "queries": entries,
    }


slow_query_result = {
    "script": profile_summary(script_profile_response),
    "range": profile_summary(range_profile_response),
    "script_count": script_count,
    "range_count": range_count,
}

print("script count:", f"{script_count:,}", "| range count:", f"{range_count:,}")
print("script profile:", json.dumps(slow_query_result["script"], indent=2))
print("range profile:", json.dumps(slow_query_result["range"], indent=2))

assert script_count == range_count
assert script_count > 0

script count: 60,014 | range count: 60,014
script profile: {
  "took_ms": 20,
  "shards": {
    "total": 1,
    "successful": 1,
    "skipped": 0,
    "failed": 0
  },
  "queries": [
    {
      "type": "ConstantScoreQuery",
      "description": "ConstantScore(ScriptQuery(Script{type=inline, lang='painless', idOrCode='doc['amount'].value >= params.min', options={}, params={min=100000}}))",
      "time_in_nanos": 20644752
    }
  ]
}
range profile: {
  "took_ms": 3,
  "shards": {
    "total": 1,
    "successful": 1,
    "skipped": 0,
    "failed": 0
  },
  "queries": [
    {
      "type": "ConstantScoreQuery",
      "description": "ConstantScore(IndexOrDocValuesQuery(indexQuery=amount:[100000 TO 2147483647], dvQuery=amount:[100000 TO 2147483647]))",
      "time_in_nanos": 274446
    }
  ]
}


## Task 10 - Create and attach an ILM policy

The policy rolls the hot index over at 50 MB or 30 days and deletes it after 90
days. The demo index has a write alias and the matching rollover-alias setting,
so ILM can manage it. Phase transitions are not forced in this lab; the policy
and the explain response are the evidence.

In [17]:
ilm_policy = {
    "phases": {
        "hot": {
            "min_age": "0ms",
            "actions": {
                "rollover": {
                    "max_primary_shard_size": "50mb",
                    "max_age": "30d",
                },
                "set_priority": {"priority": 100},
            },
        },
        "delete": {
            "min_age": "90d",
            "actions": {"delete": {}},
        },
    }
}

es.ilm.put_lifecycle(name=ILM_POLICY, policy=ilm_policy)
delete_index(ILM_DEMO)
es.indices.create(index=ILM_DEMO)
es.indices.put_alias(index=ILM_DEMO, name=ILM_ALIAS, is_write_index=True)
es.indices.put_settings(
    index=ILM_DEMO,
    settings={
        "index.lifecycle.name": ILM_POLICY,
        "index.lifecycle.rollover_alias": ILM_ALIAS,
    },
)

ilm_policy_snapshot = plain(es.ilm.get_lifecycle(name=ILM_POLICY))
ilm_explain = plain(es.ilm.explain_lifecycle(index=ILM_DEMO))
ilm_info = ilm_explain["indices"][ILM_DEMO]
ilm_wait_started = time.perf_counter()
while ilm_info.get("phase") is None and time.perf_counter() - ilm_wait_started < 60:
    time.sleep(2)
    ilm_explain = plain(es.ilm.explain_lifecycle(index=ILM_DEMO))
    ilm_info = ilm_explain["indices"][ILM_DEMO]
ilm_wait_seconds = time.perf_counter() - ilm_wait_started

print("policy names:", list(ilm_policy_snapshot.keys()))
print("ILM wait seconds:", round(ilm_wait_seconds, 2))
print("managed:", ilm_info["managed"])
print("policy:", ilm_info["policy"])
print("phase:", ilm_info.get("phase"))
print("action:", ilm_info.get("action"))
print("step:", ilm_info.get("step"))

assert ilm_info["managed"] is True
assert ilm_info["policy"] == ILM_POLICY
assert ilm_info.get("phase") == "hot"

policy names: ['p11-events-ilm']
ILM wait seconds: 2.01
managed: True
policy: p11-events-ilm
phase: hot
action: set_priority
step: set_priority


## Task 11 - Rollover and data streams

A regular rollover needs a write alias. Once the condition is met, Elasticsearch
creates the next numbered index and moves the write alias.

A data stream manages a sequence of hidden backing indices. It requires a
matching data-stream template and an `@timestamp` field. Rollover creates a new
backing index while the data stream name remains stable for writes and reads.

In [18]:
delete_index(ROLLOVER_1)
delete_index(ROLLOVER_2)
es.indices.create(
    index=ROLLOVER_1,
    settings={"number_of_shards": 1, "number_of_replicas": 0},
    mappings=MAPPINGS,
)
es.indices.put_alias(index=ROLLOVER_1, name=ROLLOVER_ALIAS, is_write_index=True)
rollover_load = bulk_load(ROLLOVER_1, count=1200, chunk_size=500)
es.indices.refresh(index=ROLLOVER_1)
rollover_response = plain(
    es.indices.rollover(
        alias=ROLLOVER_ALIAS,
        conditions={"max_docs": 1000},
        new_index=ROLLOVER_2,
    )
)
rollover_alias = plain(es.indices.get_alias(name=ROLLOVER_ALIAS))

print("rollover load:", rollover_load)
print("rollover response:", rollover_response)
print("alias contents:", {index: list(names) for index, names in rollover_alias.items()})
print("new index exists:", es.indices.exists(index=ROLLOVER_2))

assert rollover_response["rolled_over"] is True
assert rollover_response["new_index"] == ROLLOVER_2
assert es.indices.exists(index=ROLLOVER_2)
assert ROLLOVER_1 in rollover_alias and ROLLOVER_2 in rollover_alias

rollover load: {'documents': 1200, 'errors': 0, 'seconds': 0.406, 'docs_per_second': 2957.3}
rollover response: {'acknowledged': True, 'shards_acknowledged': True, 'old_index': 'p11-rollover-000001', 'new_index': 'p11-rollover-000002', 'rolled_over': True, 'dry_run': False, 'lazy': False, 'conditions': {'[max_docs: 1000]': True}}
alias contents: {'p11-rollover-000001': ['aliases'], 'p11-rollover-000002': ['aliases']}
new index exists: True


In [19]:
es.indices.put_index_template(
    name=LOGS_TEMPLATE,
    index_patterns=["p11-logs*"],
    data_stream={},
    priority=200,
    template={
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {
            "properties": {
                "@timestamp": {"type": "date"},
                "level": {"type": "keyword"},
                "message": {"type": "text"},
            }
        },
    },
)

try:
    es.indices.delete_data_stream(name=DATA_STREAM)
except NotFoundError:
    pass
es.indices.create_data_stream(name=DATA_STREAM)

for number in range(3):
    es.index(
        index=DATA_STREAM,
        document={
            "@timestamp": datetime.now(timezone.utc).isoformat(),
            "level": "info",
            "message": f"data stream message {number}",
        },
        refresh="wait_for",
    )

data_stream_before = plain(es.indices.get_data_stream(name=DATA_STREAM))["data_streams"][0]
data_stream_rollover = plain(
    es.indices.rollover(alias=DATA_STREAM, conditions={"max_docs": 1})
)
data_stream_after = plain(es.indices.get_data_stream(name=DATA_STREAM))["data_streams"][0]

before_indices = [item["index_name"] for item in data_stream_before["indices"]]
after_indices = [item["index_name"] for item in data_stream_after["indices"]]

print("data stream before:", before_indices)
print("rollover:", data_stream_rollover)
print("data stream after:", after_indices)

assert data_stream_rollover["rolled_over"] is True
assert len(after_indices) > len(before_indices)
assert all(name.startswith(".ds-p11-logs-") for name in after_indices)

data stream before: ['.ds-p11-logs-2026.11.19-000001']
rollover: {'acknowledged': True, 'shards_acknowledged': True, 'old_index': '.ds-p11-logs-2026.11.19-000001', 'new_index': '.ds-p11-logs-2026.11.19-000002', 'rolled_over': True, 'dry_run': False, 'lazy': False, 'conditions': {'[max_docs: 1]': True}}
data stream after: ['.ds-p11-logs-2026.11.19-000001', '.ds-p11-logs-2026.11.19-000002']


## Task 12 - Inspect index and node statistics

Index stats show document counts, primary store size, segments, search activity,
and indexing activity. Node stats show JVM heap and OS metrics for the local
single-node cluster. The snapshots are saved to `artifacts/stats.json`.

In [20]:
def statistics_summary(index):
    es.indices.flush(index=index)
    stats = plain(
        es.indices.stats(
            index=index,
            metric=["docs", "store", "segments", "search", "indexing"],
            level="indices",
        )
    )
    primaries = stats["indices"][index]["primaries"]
    return {
        "documents": int(primaries["docs"]["count"]),
        "primary_store_bytes": int(primaries["store"]["size_in_bytes"]),
        "segments": int(primaries["segments"]["count"]),
        "search_query_total": int(primaries["search"]["query_total"]),
        "search_query_time_ms": int(primaries["search"]["query_time_in_millis"]),
        "indexing_index_total": int(primaries["indexing"]["index_total"]),
        "indexing_index_time_ms": int(primaries["indexing"]["index_time_in_millis"]),
    }


index_stats = {
    index: statistics_summary(index)
    for index in [BASELINE, OPTIMIZED, REINDEXED]
}
node_stats_response = plain(
    es.nodes.stats(metric=["jvm", "os", "indices"])
)
node_summary = {}
for node_id, node in node_stats_response["nodes"].items():
    node_summary[node_id] = {
        "name": node.get("name"),
        "heap_used_in_bytes": node.get("jvm", {}).get("mem", {}).get("heap_used_in_bytes"),
        "heap_max_in_bytes": node.get("jvm", {}).get("mem", {}).get("heap_max_in_bytes"),
        "cpu_percent": node.get("os", {}).get("cpu", {}).get("percent"),
        "documents": node.get("indices", {}).get("docs", {}).get("count"),
        "store_bytes": node.get("indices", {}).get("store", {}).get("size_in_bytes"),
    }

stats_snapshot = {
    "cluster_health": plain(es.cluster.health()),
    "indices": index_stats,
    "nodes": node_summary,
}
print(json.dumps(stats_snapshot, indent=2))

assert index_stats[BASELINE]["documents"] == TOTAL
assert index_stats[OPTIMIZED]["documents"] == TOTAL
assert index_stats[REINDEXED]["documents"] == TOTAL
assert node_summary

{
  "cluster_health": {
    "cluster_name": "my-elasticsearch-cluster",
    "status": "yellow",
    "timed_out": false,
    "number_of_nodes": 1,
    "number_of_data_nodes": 1,
    "active_primary_shards": 72,
    "active_shards": 72,
    "relocating_shards": 0,
    "initializing_shards": 0,
    "unassigned_shards": 2,
    "unassigned_primary_shards": 0,
    "delayed_unassigned_shards": 0,
    "number_of_pending_tasks": 0,
    "number_of_in_flight_fetch": 0,
    "task_max_waiting_in_queue_millis": 0,
    "active_shards_percent_as_number": 97.2972972972973
  },
  "indices": {
    "p11-baseline-events": {
      "documents": 100000,
      "primary_store_bytes": 19375515,
      "segments": 23,
      "search_query_total": 1235,
      "search_query_time_ms": 1129,
      "indexing_index_total": 100000,
      "indexing_index_time_ms": 30480
    },
    "p11-events-optimized": {
      "documents": 100000,
      "primary_store_bytes": 8895089,
      "segments": 1,
      "search_query_total": 150,

## Final report

The JSON artifacts contain the complete measured results. Performance numbers
are from one local Docker node with zero replicas; they are useful for relative
comparison, not as production capacity figures.

In [21]:
baseline_metrics = index_metrics(BASELINE)
optimized_metrics = index_metrics(OPTIMIZED)
shard_1_metrics = index_metrics(SHARDS_1)
shard_3_metrics = index_metrics(SHARDS_3)
baseline_store_bytes = baseline_metrics["primary_store_bytes"]
optimized_store_bytes = optimized_metrics["primary_store_bytes"]
store_reduction = round(
    (baseline_store_bytes - optimized_store_bytes) / baseline_store_bytes * 100, 2
)
indexing_change = round(
    (optimized_load["docs_per_second"] - baseline_load["docs_per_second"])
    / baseline_load["docs_per_second"]
    * 100,
    2,
)

print("Final baseline metrics:", baseline_metrics)
print("Final optimized metrics:", optimized_metrics)
print("Final store reduction:", store_reduction, "%")

index_report = {
    "environment": {
        "elasticsearch_version": info["version"]["number"],
        "cluster_name": info["cluster_name"],
        "node_name": info["name"],
        "baseline_index": BASELINE,
        "optimized_index": OPTIMIZED,
    },
    "dataset": {
        "documents": TOTAL,
        "small_experiment_documents": SMALL_TOTAL,
        "fields": FIELDS,
    },
    "templates": {
        "components": component_snapshot,
        "index_template": template_snapshot,
    },
    "alias": {
        "name": READ_ALIAS,
        "before": alias_before,
        "after": alias_after,
        "count_before": baseline_alias_count,
        "count_after": optimized_alias_count,
    },
    "reindex": {
        "source": BASELINE,
        "destination": REINDEXED,
        "seconds": round(reindex_seconds, 3),
        "response": reindex_response,
        "documents": reindexed_count,
    },
    "shards": {
        "one": {
            "load": shard_1_load,
            "metrics": shard_1_metrics,
            "query": shard_1_query,
            "count": shard_1_count,
        },
        "three": {
            "load": shard_3_load,
            "metrics": shard_3_metrics,
            "query": shard_3_query,
            "count": shard_3_count,
        },
    },
    "refresh": {
        "one_second": refresh_1s_load,
        "disabled_during_load": refresh_off_load,
        "speedup": round(
            refresh_off_load["docs_per_second"] / refresh_1s_load["docs_per_second"], 3
        ),
    },
    "slow_query": slow_query_result,
    "ilm": {
        "policy": ILM_POLICY,
        "policy_snapshot": ilm_policy_snapshot,
        "explain": ilm_explain,
    },
    "rollover": {
        "load": rollover_load,
        "response": rollover_response,
        "alias": rollover_alias,
    },
    "data_stream": {
        "name": DATA_STREAM,
        "before": before_indices,
        "after": after_indices,
        "rollover": data_stream_rollover,
    },
}

optimization_report = {
    "baseline": {
        "index": BASELINE,
        "shards": int(index_setting(BASELINE, "number_of_shards")),
        "mapping": "dynamic",
        "refresh_interval_during_load": "1s",
        "load": baseline_load,
        "metrics": baseline_metrics,
    },
    "optimized": {
        "index": OPTIMIZED,
        "shards": int(index_setting(OPTIMIZED, "number_of_shards")),
        "mapping": "explicit strict",
        "refresh_interval_during_load": "-1",
        "codec": index_setting(OPTIMIZED, "codec"),
        "load": optimized_load,
        "force_merge_seconds": round(force_merge_seconds, 3),
        "metrics": optimized_metrics,
    },
    "reindex": {
        "source": BASELINE,
        "destination": REINDEXED,
        "seconds": round(reindex_seconds, 3),
        "documents": reindexed_count,
    },
    "queries": query_comparison,
    "changes": {
        "indexing_throughput_percent": indexing_change,
        "primary_store_reduction_percent": store_reduction,
        "segments_before": baseline_metrics["segments"],
        "segments_after": optimized_metrics["segments"],
    },
}

benchmark_report = {
    "indexing": {
        "baseline": baseline_load,
        "optimized": optimized_load,
        "refresh_1s": refresh_1s_load,
        "refresh_disabled": refresh_off_load,
    },
    "shards": index_report["shards"],
    "queries": query_comparison,
    "slow_query": slow_query_result,
}

(ARTIFACTS / "index_report.json").write_text(
    json.dumps(plain(index_report), indent=2), encoding="utf-8"
)
(ARTIFACTS / "optimization_report.json").write_text(
    json.dumps(plain(optimization_report), indent=2), encoding="utf-8"
)
(ARTIFACTS / "benchmark.json").write_text(
    json.dumps(plain(benchmark_report), indent=2), encoding="utf-8"
)
(ARTIFACTS / "stats.json").write_text(
    json.dumps(plain(stats_snapshot), indent=2), encoding="utf-8"
)

print("Final counts:")
print("  baseline:", f"{index_count(BASELINE):,}")
print("  optimized:", f"{index_count(OPTIMIZED):,}")
print("  reindexed:", f"{index_count(REINDEXED):,}")
print("  alias:", READ_ALIAS, "->", index_count(READ_ALIAS))
print("Wrote:")
for name in ["index_report.json", "optimization_report.json", "benchmark.json", "stats.json"]:
    print("  ", ARTIFACTS / name)
print("All Practical 11 assertions passed.")

Final baseline metrics: {'documents': 100000, 'primary_store_bytes': 19375515, 'segments': 23, 'search_query_total': 1235, 'search_query_time_ms': 1129, 'indexing_index_total': 100000, 'indexing_index_time_ms': 30480}
Final optimized metrics: {'documents': 100000, 'primary_store_bytes': 8895089, 'segments': 1, 'search_query_total': 150, 'search_query_time_ms': 122, 'indexing_index_total': 100000, 'indexing_index_time_ms': 5373}
Final store reduction: 54.09 %
Final counts:
  baseline: 100,000
  optimized: 100,000


  reindexed: 100,000
  alias: p11-events-read -> 100000
Wrote:
   C:\D-Drive\Elastic Search Practicals\Practical_11\artifacts\index_report.json
   C:\D-Drive\Elastic Search Practicals\Practical_11\artifacts\optimization_report.json
   C:\D-Drive\Elastic Search Practicals\Practical_11\artifacts\benchmark.json
   C:\D-Drive\Elastic Search Practicals\Practical_11\artifacts\stats.json
All Practical 11 assertions passed.


## Summary

- Component templates make reusable settings and mappings; an index template
  composes them for a family of indices.
- An alias provides a stable read name. Updating aliases in one request avoids a
  visible missing-alias window.
- Reindexing copies existing data into a new index with new settings or
  mappings.
- One shard is usually the right default for a modest single-node index.
  Additional shards add overhead without adding hardware.
- Disabling refresh during a bulk load removes unnecessary refresh work.
- `best_compression` and a force-merge to one segment reduce storage and
  segment overhead, but their benefit depends on the data and workload.
- Filter queries are preferable to per-document scripts for selectivity.
- ILM, rollover, and data streams automate long-term index management.
- Index and node statistics provide the evidence behind optimization decisions.